## Text-based Search

In [16]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENSEARCH_USER = os.getenv("OPENSEARCH_USER")
OPENSEARCH_PASSWORD = os.getenv("OPENSEARCH_PASSWORD")
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST")
OPENSEARCH_PORT = os.getenv("OPENSEARCH_PORT")

In [17]:
import pprint as pp
from opensearchpy import OpenSearch
from opensearchpy import helpers

index_name = OPENSEARCH_USER + '_project'
# Create the client with SSL/TLS enabled, but hostname verification disabled.
client = OpenSearch(
    hosts = [{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_compress = True, # enables gzip compression for request bodies
    http_auth = (OPENSEARCH_USER, OPENSEARCH_PASSWORD),
    use_ssl = True,
    url_prefix = 'opensearch_v3',
    verify_certs = False,
    ssl_assert_hostname = False,
    ssl_show_warn = False
)

if client.indices.exists(index=index_name):

    resp = client.indices.open(index=index_name)
    print(resp)

    print('\n----------------------------------------------------------------------------------- INDEX SETTINGS')
    settings = client.indices.get_settings(index=index_name)
    pp.pprint(settings)

    print('\n----------------------------------------------------------------------------------- INDEX MAPPINGS')
    mappings = client.indices.get_mapping(index=index_name)
    pp.pprint(mappings)

    print('\n----------------------------------------------------------------------------------- INDEX #DOCs')
    print(client.count(index=index_name))
else:
    print("Index does not exist.")


{'acknowledged': True, 'shards_acknowledged': True}

----------------------------------------------------------------------------------- INDEX SETTINGS
{'uservl07_project': {'settings': {'index': {'creation_date': '1774740542646',
                                             'knn': 'true',
                                             'knn.derived_source': {'enabled': 'true'},
                                             'number_of_replicas': '0',
                                             'number_of_shards': '4',
                                             'provided_name': 'uservl07_project',
                                             'refresh_interval': '1s',
                                             'replication': {'type': 'DOCUMENT'},
                                             'similarity': {'bm25-0-75': {'b': '0.75',
                                                                          'k1': '0.0',
                                                                      

In [18]:
from datasets import load_dataset
import aiohttp

ds = load_dataset("HuggingFaceM4/COCO", split="validation",
                  storage_options={'client_kwargs': {'timeout': aiohttp.ClientTimeout(total=36000)}}
                  )
print(ds.column_names)
pp.pprint(ds[0])
print(ds[0]['sentences']['raw'])
print(ds[0]['image'].width)


['image', 'filepath', 'sentids', 'filename', 'imgid', 'split', 'sentences', 'cocoid']
{'cocoid': 184613,
 'filename': 'COCO_val2014_000000184613.jpg',
 'filepath': 'COCO_val2014_000000184613.jpg',
 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x336 at 0x16884B0B0>,
 'imgid': 2,
 'sentences': {'imgid': 2,
               'raw': 'A child holding a flowered umbrella and petting a yak.',
               'sentid': 474921,
               'tokens': ['a',
                          'child',
                          'holding',
                          'a',
                          'flowered',
                          'umbrella',
                          'and',
                          'petting',
                          'a',
                          'yak']},
 'sentids': [474921, 479322, 479334, 481560, 483594],
 'split': 'val'}
A child holding a flowered umbrella and petting a yak.
500


Set up an OpenSearch index and define mappings for image identifiers,
captions, and COCO metadata (categories, image dimensions).

In [ ]:
Delete index if necessary

pp.pprint(client.indices.delete(index=index_name))

{'acknowledged': True}


In [21]:
index_body = {
   "settings":{
      "index":{
         "number_of_replicas":0,
         "number_of_shards":4,
         "refresh_interval":"-1",
         "knn":"true"
      },
      "similarity":{
          "bm25-20-75":{
              "type":"BM25",
              "k1":2.0,
              "b":0.75
          },
          "bm25-12-0":{
              "type":"BM25",
              "k1":1.2,
              "b":0.0
          },
          "bm25-0-75":{
              "type":"BM25",
              "k1":0.0,
              "b":0.75
          }

      }
   },
   "mappings":{
       "dynamic":      "strict",
       "properties":{
            #"_id": { "type": "keyword"},
            "image_id": { "type": "keyword"},
            "caption": { 
                "type": "text",
                "analyzer": "standard",
                "similarity":"BM25",
                "fields":{
                    "bm25_20_75": {
                        "type": "text",
                        "analyzer": "standard",
                        "similarity":"bm25-20-75"
                    },
                    "bm25_12_0": {
                        "type": "text",
                        "analyzer": "standard",
                        "similarity":"bm25-12-0"
                    },
                    "bm25_0_75": {
                        "type": "text",
                        "analyzer": "standard",
                        "similarity":"bm25-0-75"
                     }
                  },
            },
            "categories": { "type": "keyword"},
            "width": { "type": "integer"},
            "height": { "type": "integer"},

            "SBERT_embedding": {
                "type":"knn_vector",
                "dimension": 768,
                "method":{
                    "name":"hnsw",
                    "space_type":"innerproduct",
                    "engine":"faiss",
                    "parameters":{
                        "ef_construction":256,
                        "m":48
                    }
                },
            },

            "BGE_embedding": {
                "type":"knn_vector",
                "dimension": 384,
                "method":{
                    "name":"hnsw",
                    "space_type":"innerproduct",
                    "engine":"faiss",
                    "parameters":{
                        "ef_construction":256,
                        "m":48
                    }
                }
            }
      }
   }
}

if client.indices.exists(index=index_name):
    print("Index already existed. Nothing to be done.")
else:        
    response = client.indices.create(index=index_name, body=index_body)
    print('\nCreating index:')
    print(response)



Creating index:
{'acknowledged': True, 'shards_acknowledged': True, 'index': 'uservl07_project'}


Load dataset captions in index

In [22]:
def index_data(client, index_name, dataset):
    actions= []
    for i,item in enumerate(dataset):
        action={
            "_id":item['sentences']['sentid'],
            "image_id":item['sentences']['imgid'],
            "caption":item['sentences']['raw'],
            #"categories":
            "width":item['image'].width,
            "height":item['image'].height
        }
        actions.append(action)
        if len(actions) == 1000:
            helpers.bulk(client, actions, index=index_name)
            print(f"Indexed {i+1} documents")
            actions = []
    if actions:
        helpers.bulk(client, actions, index=index_name)
        print(f"Indexed {len(dataset)} documents in total")
count = client.count(index=index_name)
print(count['count'])

if count['count'] == 0:
    index_data(client, index_name, ds)
else:
    print("Index already contains documents. Nothing to be done.")

0
Indexed 1000 documents
Indexed 2000 documents
Indexed 3000 documents
Indexed 4000 documents
Indexed 5000 documents
Indexed 6000 documents
Indexed 7000 documents
Indexed 8000 documents
Indexed 9000 documents
Indexed 10000 documents
Indexed 11000 documents
Indexed 12000 documents
Indexed 13000 documents
Indexed 14000 documents
Indexed 15000 documents
Indexed 16000 documents
Indexed 17000 documents
Indexed 18000 documents
Indexed 19000 documents
Indexed 20000 documents
Indexed 21000 documents
Indexed 22000 documents
Indexed 23000 documents
Indexed 24000 documents
Indexed 25000 documents
Indexed 25010 documents in total


In [23]:
client.indices.put_settings(index=index_name, body={"index": {"refresh_interval": "1s"}}) #set refresh interval to 1s to make all operations available for search after 1s
client.indices.refresh(index=index_name) #refresh immediately

{'_shards': {'total': 4, 'successful': 4, 'failed': 0}}

In [24]:

count = client.count(index=index_name)
print(count['count'])

# check a random document
result = client.search(
    index=index_name,
    body={"query": {"match_all": {}}, "size": 1}
)
print(result['hits']['hits'][0]['_source'])

25010
{'image_id': 2, 'caption': 'A child holding a flowered umbrella and petting a yak.', 'width': 500, 'height': 336}


Perform the same query computing similarity with BM25 with different parameters.
For field boosting, idea is to use another field, but the dataset doesn't have a categories objects

In [28]:
query_txt = ds[0]['sentences']['raw']


bm25_fields = ["caption", "caption.bm25_20_75", "caption.bm25_12_0", "caption.bm25_0_75"]

for field in bm25_fields:
    query ={
        "size": 5,
        "query": {
            "match": {
                field: {
                    "query": query_txt
                }
            }
        }
    }
    response = client.search(body=query, index=index_name)
    #pp.pprint(response)
    print(f'\nSearch results for field "{field}":')
    for hit in response['hits']['hits']:
        print(f"Score: {hit['_score']:.4f}, Doc ID: {hit['_id']}, Caption: {hit['_source']['caption']}")


Search results for field "caption":
Score: 17.1729, Doc ID: 474921, Caption: A child holding a flowered umbrella and petting a yak.
Score: 5.7630, Doc ID: 192463, Caption: a close up of a child holding a closed umbrella
Score: 5.7511, Doc ID: 237293, Caption: A woman with child pack on holding an umbrella.
Score: 5.3220, Doc ID: 402327, Caption: A woman with a black umbrella and a child with a red and black umbrella.
Score: 5.2975, Doc ID: 824983, Caption: Someone is petting a horse while holding a baby.

Search results for field "caption.bm25_20_75":
Score: 12.6749, Doc ID: 474921, Caption: A child holding a flowered umbrella and petting a yak.
Score: 4.2757, Doc ID: 237293, Caption: A woman with child pack on holding an umbrella.
Score: 4.2714, Doc ID: 192463, Caption: a close up of a child holding a closed umbrella
Score: 4.0175, Doc ID: 402327, Caption: A woman with a black umbrella and a child with a red and black umbrella.
Score: 3.9538, Doc ID: 824983, Caption: Someone is petti

## Semantic Text Search

In [29]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd


df = ds.to_pandas()
#raw_captions = df['sentences'].apply(lambda x: x['raw']).head()

df = pd.DataFrame(
    {
        "sentid": df['sentences'].apply(lambda x: x['sentid']),
        "imgid": df['sentences'].apply(lambda x: x['imgid']),
        "raw": df['sentences'].apply(lambda x: x['raw']),
    }
)
df.head()

sbert_model = SentenceTransformer('all-mpnet-base-v2')
bge_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
sbert_encoding = sbert_model.encode(df['raw'], show_progress_bar=True)
bge_encoding = bge_model.encode(df['raw'], show_progress_bar=True)

print("SBERT Encodings shape:", sbert_encoding.shape)
print("BGE Encodings shape:", bge_encoding.shape)

df['sbert_embedding'] = list(sbert_encoding)
df['bge_embedding'] = list(bge_encoding)

df.head()
df.to_parquet('coco_captions_embeddings.parquet', index=False)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/782 [00:00<?, ?it/s]

Batches:   0%|          | 0/782 [00:00<?, ?it/s]

SBERT Encodings shape: (25010, 768)
BGE Encodings shape: (25010, 384)


In [30]:
pp.pprint(df.head())

   sentid  imgid                                                raw  \
0  474921      2  A child holding a flowered umbrella and pettin...   
1  479322      2  A young man holding an umbrella next to a herd...   
2  479334      2  a young boy barefoot holding an umbrella touch...   
3  481560      2  A young boy with an umbrella who is touching t...   
4  483594      2  A boy holding an umbrella while standing next ...   

                                     sbert_embedding  \
0  [0.043276526, -0.07047357, 0.005368873, 0.0237...   
1  [0.006170751, -0.051451195, 0.015064564, 0.029...   
2  [-0.004062805, -0.0505614, 0.010549658, 0.0267...   
3  [0.0056119883, -0.055808183, -0.00492116, 0.01...   
4  [-0.002827854, -0.02579093, -0.016782232, 0.03...   

                                       bge_embedding  
0  [0.011913764, -0.0007062534, 0.08118155, -0.02...  
1  [0.029823745, 0.016038753, 0.009575873, -0.024...  
2  [0.009185313, 0.03758348, 0.016738353, 0.00086...  
3  [0.0020447166

## Update index


In [31]:
df = pd.read_parquet('coco_captions_embeddings.parquet')
df.head()

,sentid,imgid,raw,sbert_embedding,bge_embedding
0,474921,2,A child holding a flowered umbrella and pettin...,"[0.043276526, -0.07047357, 0.005368873, 0.0237...","[0.011913764, -0.0007062534, 0.08118155, -0.02..."
1,479322,2,A young man holding an umbrella next to a herd...,"[0.006170751, -0.051451195, 0.015064564, 0.029...","[0.029823745, 0.016038753, 0.009575873, -0.024..."
2,479334,2,a young boy barefoot holding an umbrella touch...,"[-0.004062805, -0.0505614, 0.010549658, 0.0267...","[0.009185313, 0.03758348, 0.016738353, 0.00086..."
3,481560,2,A young boy with an umbrella who is touching t...,"[0.0056119883, -0.055808183, -0.00492116, 0.01...","[0.0020447166, 0.02044127, 0.030630918, -0.031..."
4,483594,2,A boy holding an umbrella while standing next ...,"[-0.002827854, -0.02579093, -0.016782232, 0.03...","[0.039909482, 0.03083807, 0.012112419, 0.01105..."


In [32]:
def update_index(client, index_name, df):
    actions = []
    for i, row in df.iterrows():
        action = {
            "_op_type": "update",
            "_index": index_name,
            "_id": row['sentid'],
            "doc": {
                "SBERT_embedding": row['sbert_embedding'],
                "BGE_embedding": row['bge_embedding']
            }
        }
        actions.append(action)
        if len(actions) == 1000:
            helpers.bulk(client, actions)
            print(f"Updated {i+1} documents")
            actions = []
    if actions:
        helpers.bulk(client, actions)
        print(f"Updated {len(df)} documents in total")

In [33]:

update_index(client, index_name, df)

Updated 1000 documents
Updated 2000 documents
Updated 3000 documents
Updated 4000 documents
Updated 5000 documents
Updated 6000 documents
Updated 7000 documents
Updated 8000 documents
Updated 9000 documents
Updated 10000 documents
Updated 11000 documents
Updated 12000 documents
Updated 13000 documents
Updated 14000 documents
Updated 15000 documents
Updated 16000 documents
Updated 17000 documents
Updated 18000 documents
Updated 19000 documents
Updated 20000 documents
Updated 21000 documents
Updated 22000 documents
Updated 23000 documents
Updated 24000 documents
Updated 25000 documents
Updated 25010 documents in total


In [35]:
#query a random document to check if the update was successful
result = client.search(
    index=index_name,
    body={"query": {"match_all": {}}, "size": 1}
)
pp.pprint(result['hits']['hits'][0]['_source'])

{'BGE_embedding': [0.015136986,
                   -0.05424819,
                   -0.0054895715,
                   -0.013947624,
                   -0.008411746,
                   0.020166786,
                   0.039090257,
                   0.0135464445,
                   -0.017640308,
                   0.050281048,
                   -0.062031772,
                   -0.046873767,
                   0.006212198,
                   0.011452262,
                   -0.008550767,
                   0.028524283,
                   0.024093652,
                   -0.05105247,
                   -0.04480061,
                   0.0030852354,
                   0.038627088,
                   0.01747171,
                   0.0013991954,
                   -0.04686673,
                   -0.025517324,
                   0.031503256,
                   -0.014248372,
                   0.01901407,
                   -0.03153394,
                   -0.08347266,
                   -0.0564242

In [36]:
client.indices.close(index=index_name)

{'acknowledged': True,
 'shards_acknowledged': True,
 'indices': {'uservl07_project': {'closed': True}}}